# Notebook 1 — Download de dados do GWOSC

Neste notebook, você localizará dados públicos do **Gravitational-Wave Open Science Center (GWOSC)** 
e fará o download de um arquivo HDF5 (*Hierarchical Data Format, versão 5*) do detector LIGO Hanford.

## Objetivos

Ao final da atividade, você deverá ser capaz de:

- identificar períodos com dados disponíveis no GWOSC;
- consultar o acervo por meio de uma API;
- baixar um arquivo de *strain* para uso nos próximos notebooks.

> **Como trabalhar:** execute as células na ordem. No Colab, salve antes uma cópia pessoal do notebook. Não é necessária GPU.

**Origem:** material adaptado de [Introduction to GWOSC Data](https://github.com/gwosc-tutorial/introduction_gwosc_data), distribuído sob licença GPL-3.0 ou posterior.

**Tradução e organização:** Ronai Lisbôa — UFRN.


# 1. Preparação para importar as bibliotecas


In [ ]:
# requests é uma biblioteca utilizada para realizar requisições pela internet.
# Neste curso, ela será usada para consultar a API do GWOSC e baixar arquivos.
import requests

# Path é uma classe da biblioteca padrão do Python usada para representar
# caminhos, pastas e arquivos de maneira independente do sistema operacional.
from pathlib import Path

# Exibe a versão instalada da biblioteca requests.
print("Versão do requests:", requests.__version__)


# GWpy é uma biblioteca Python desenvolvida para acessar, representar,
# visualizar e analisar dados de detectores de ondas gravitacionais.
import gwpy

# TimeSeries é uma classe do GWpy que representa dados medidos ao longo do tempo,
# como a série temporal de strain dos detectores LIGO e Virgo.
from gwpy.timeseries import TimeSeries

# Exibe a versão instalada da biblioteca GWpy.
print("Versão do GWpy:", gwpy.__version__)

## 1. Localizar dados disponíveis

O GWOSC organiza os dados por campanhas de observação, detectores e intervalos de tempo.

1. Acesse a página [Observatory Data Sets](https://gwosc.org/data/).
2. Localize **O1 Data Release**.
3. Abra a [timeline da campanha O1](https://gwosc.org/timeline/show/O1/H1_DATA*L1_DATA/1126051217/11203200/).

A *flag* `DATA` indica os intervalos em que há dados disponibilizados. Ela não garante que o intervalo esteja livre de ruído; a qualidade dos dados será estudada no Notebook 3.


![Timeline da campanha O1](https://raw.githubusercontent.com/Ronai/gwd/main/introduction_gwosc_data/img/o1-timeline.png)

[Abrir a imagem em tamanho completo](https://raw.githubusercontent.com/Ronai/gwd/main/introduction_gwosc_data/img/o1-timeline.png)

### Observe a timeline

- `H1` representa o detector LIGO Hanford.
- Uma barra indica que havia dados disponíveis naquele instante; a ausência de barra indica indisponibilidade.
- É possível ampliar e deslocar o gráfico usando o mouse.


## 2. Escolher um arquivo do acervo

Vamos procurar dados do detector **H1** referentes a 14 de setembro de 2015.

1. Na página [Observatory Data Sets](https://gwosc.org/data/), localize **O1 Data Release** e selecione **4 kHz Data**.
2. Escolha o detector `H1`.
3. Informe o intervalo:
   - início: `2015-09-14 00:00:00 UTC` — GPS `1126224017`;
   - fim: `2015-09-14 23:59:59 UTC` — GPS `1126310416`.
4. Selecione o formato **Time series data in HDF5 and Frame files**.

Para converter entre UTC e GPS, utilize a [ferramenta de conversão de tempo do GWOSC](https://gwosc.org/gps/).


![Arquivos disponíveis no acervo O1](https://raw.githubusercontent.com/Ronai/gwd/main/introduction_gwosc_data/img/o1-dataset.png)

[Abrir a imagem em tamanho completo](https://raw.githubusercontent.com/Ronai/gwd/main/introduction_gwosc_data/img/o1-dataset.png)

Cada linha representa um arquivo, também chamado de *tile*, que cobre **4096 segundos**. A coluna de *duty cycle* informa qual porcentagem desse intervalo contém dados do modo científico.

Nesta atividade, usaremos o arquivo HDF5 iniciado no GPS `1126256640`:

```text
H-H1_LOSC_4_V1-1126256640-4096.hdf5
```


### Onde o arquivo será salvo?

- **JupyterLab:** o arquivo ficará na pasta de execução do notebook. Mantenha os notebooks da sequência nessa mesma pasta.
- **Google Colab:** o arquivo ficará no armazenamento temporário da sessão. Uma nova sessão poderá exigir outro download.

Salve uma cópia do notebook no Drive ou baixe o arquivo `.ipynb` para preservar suas respostas.


## 3. Fazer o download com Python

O GWOSC oferece uma API que retorna informações sobre os arquivos disponíveis em formato JSON. Primeiro, definiremos uma função para consultar o acervo.


In [ ]:
def fetch_strain_list(run, detector, gps_start, gps_end):
    """Consulta os arquivos de strain disponíveis no GWOSC."""

    url = (
        f"https://gwosc.org/archive/links/{run}/{detector}/"
        f"{gps_start}/{gps_end}/json/"
    )

    response = requests.get(url, timeout=(15, 120))
    response.raise_for_status()       # Interrompe a execução se ocorrer erro HTTP.

    return response.json()["strain"]  # Converte o JSON e retorna a lista de arquivos.


### 3.1 Consultar os arquivos da campanha O1

A próxima célula consulta o intervalo de um dia e apresenta somente os registros em formato HDF5.


In [ ]:
strain_files = fetch_strain_list(
    run="O1",
    detector="H1",
    gps_start=1126224017,
    gps_end=1126310416,
)

hdf5_files = [item for item in strain_files if item["format"] == "hdf5"]

print(f"Foram encontrados {len(hdf5_files)} arquivos HDF5.\n")

for item in hdf5_files:
    print(item["GPSstart"], item["duty_cycle"], item["url"])


### 3.2 Baixar o arquivo selecionado

Agora selecionaremos o arquivo que começa no GPS `1126256640`, em formato HDF5 e com taxa de amostragem de `4096 Hz`.


In [ ]:
def download_strain_file(download_url):
    """Baixa o arquivo somente se ele ainda não existir."""

    filename = Path(download_url.split("/")[-1])

    # Evita repetir o download
    if filename.exists():
        print(f"Arquivo já existente: {filename}")
        return str(filename)

    print(f"Baixando: {download_url}")

    with requests.get(
        download_url,
        stream=True,
        timeout=(15, 120)
    ) as response:

        response.raise_for_status()



        with open(filename, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)

    print(f"Download concluído: {filename}")
    return str(filename)



candidates = [
    item for item in strain_files
    if item["GPSstart"] == 1126256640
    and item["format"] == "hdf5"
    and item["sampling_rate"] == 4096
]

if not candidates:
    raise RuntimeError("O arquivo esperado não foi encontrado no GWOSC.")

fname = download_strain_file(candidates[0]["url"])

print(f"Pasta dos dados: {Path(fname).resolve().parent}")

O arquivo `H-H1_LOSC_4_V1-1126256640-4096.hdf5` contém **4096 segundos de dados**, amostrados a **4096 Hz**, e será utilizado no Notebook 2.

> **No Colab:** mantenha os notebooks na mesma sessão de execução ou faça novamente o download no próximo notebook. A execução repetida da célula de download substitui o arquivo existente de mesmo nome.


> **Boas práticas de acesso aos dados**
>
> Evite baixar grandes quantidades de arquivos em um laço. Isso pode sobrecarregar os servidores do GWOSC e tornar o processo instável. Para volumes maiores, consulte a documentação sobre [acesso aos dados por CVMFS](https://computing.docs.ligo.org/guide/compsoft/roadmap/LVK/cvmfs_for_data/?h=cvmfs).


## 4. Verificação e exercícios

1. Qual detector, campanha de observação, taxa de amostragem e duração correspondem ao arquivo baixado?
2. Quantas amostras são esperadas em 4096 segundos, com taxa de amostragem de 4096 Hz? Faça o cálculo com Python.
3. Execute novamente a célula de download. O que acontece com o arquivo que já estava na pasta?

> Registre as respostas em uma nova célula de texto. A duração do arquivo não corresponde à duração de um sinal de coalescência.


## Próximo passo

No **Notebook 2**, você aprenderá a abrir o arquivo HDF5, examinar sua estrutura e acessar a série temporal de *strain*.
